<a href="https://colab.research.google.com/github/LFernandez898/evparcial1/blob/main/Notebook_Limpieza_FIFA21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Carga y Exploración Inicial

In [26]:
import pandas as pd
import numpy as np

# Cargamos el dataset original
# low_memory=False evita advertencias por el tamaño del archivo
df = pd.read_csv('fifa21_raw_data.csv', low_memory=False)

# Revisamos qué columnas tienen nulos y qué tipo de datos son
print("Información del Dataset:")
df.info()

Información del Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18979 entries, 0 to 18978
Data columns (total 77 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   photoUrl          18979 non-null  object
 1   LongName          18979 non-null  object
 2   playerUrl         18979 non-null  object
 3   Nationality       18979 non-null  object
 4   Positions         18979 non-null  object
 5   Name              18979 non-null  object
 6   Age               18979 non-null  int64 
 7   ↓OVA              18979 non-null  int64 
 8   POT               18979 non-null  int64 
 9   Team & Contract   18979 non-null  object
 10  ID                18979 non-null  int64 
 11  Height            18979 non-null  object
 12  Weight            18979 non-null  object
 13  foot              18979 non-null  object
 14  BOV               18979 non-null  int64 
 15  BP                18979 non-null  object
 16  Growth            18979 non-null 

2. Funciones de Limpieza (Requisito: Código Modular)

In [22]:
def limpiar_altura(valor):
    if '\"' in str(valor): # Si viene en pies/pulgadas
        pies, pulg = valor.split("'")
        pulg = pulg.replace('"', '')
        return round((int(pies) * 30.48) + (int(pulg) * 2.54), 2)
    return float(str(valor).replace('cm', ''))

# Aplicar de forma eficiente (Vectorización)
df['Height'] = df['Height'].apply(limpiar_altura)

In [27]:
# Función para limpiar el dinero (Símbolos € y letras K o M)
def limpiar_dinero(valor):
    if isinstance(valor, str):
        valor = valor.replace('€', '')
        if 'M' in valor:
            return float(valor.replace('M', '')) * 1000000
        elif 'K' in valor:
            return float(valor.replace('K', '')) * 1000
    return float(valor)

# Función para limpiar la altura (algunos vienen en cm y otros en pies/pulgadas)
def limpiar_altura(valor):
    valor = str(valor)
    if "'" in valor:  # Si está en formato pies'pulgadas"
        pies, pulgadas = valor.split("'")
        pulgadas = pulgadas.replace('"', '')
        # Convertimos a cm (1 pie = 30.48cm, 1 pulgada = 2.54cm)
        return round((int(pies) * 30.48) + (int(pulgadas) * 2.54), 1)
    return float(valor.replace('cm', ''))

# Aplicamos las funciones a las columnas correspondientes
df['Value'] = df['Value'].apply(limpiar_dinero)
df['Wage'] = df['Wage'].apply(limpiar_dinero)
df['Height'] = df['Height'].apply(limpiar_altura)

print("Limpieza de dinero y altura completada.")

Limpieza de dinero y altura completada.


3. Transformación con Pipeline (Requisito Crítico para el 30%)

In [29]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Seleccionamos las columnas que queremos procesar
cols_numericas = ['Age', '↓OVA', 'Height', 'Value', 'Wage']
cols_categoricas = ['foot', 'BP']

# 1. Pipeline para números: Llena nulos con el promedio y escala los datos
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# 2. Pipeline para texto: Llena nulos y convierte a números (OneHot)
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Unimos ambos procesos en un solo transformador
preprocesador = ColumnTransformer([
    ('num', num_pipeline, cols_numericas),
    ('cat', cat_pipeline, cols_categoricas)
])

# Ejecutamos el Pipeline
datos_listos = preprocesador.fit_transform(df)
print("Pipeline ejecutado: Datos limpios, sin nulos y escalados.")

Pipeline ejecutado: Datos limpios, sin nulos y escalados.
